# Musanga V3 — Notebook 01: five-mosaic / 30-circle reference registry

**Current study design:** MOS 3–4 is excluded entirely. Baego, MOS 1–2, MOS 5–6, Yangambi and Yoko remain, with 30 exhaustive circles. All original geometry files are left untouched; filtered derived reference outputs are written to a new folder. NB00 still audits *all raw inputs*, including excluded MOS 3–4.


## 01 — Drive environment and audited input registry


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib
from pathlib import Path
import json, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from rasterio.windows import Window, from_bounds
from rasterio.features import rasterize
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
DATA=next((p for p in [DRIVE/'Colab Notebooks/Musanga/Final/Data',DRIVE/'Musanga/Final/Data'] if p.exists()),None)
assert DATA is not None,'Final/Data directory not found'
AUDIT=DATA.parent/'Outputs/V3/00_input_audit'
OUT=DATA.parent/'Outputs/V3/01_reference_data_no_MOS34';OUT.mkdir(parents=True,exist_ok=True)
manifest_path=AUDIT/'00_input_manifest.json'
assert manifest_path.exists(),'Run Notebook 00 through Block 10 first'
A=json.loads(manifest_path.read_text())
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
EXCLUDED_IDS={6,9,24,25}
assert EXCLUDED_MOSAIC in [r['file'] for r in A['mosaics']]
MOSAICS={r['file']:r for r in A['mosaics'] if r['file']!=EXCLUDED_MOSAIC}
assert A.get('schema_version')=='v3_input_audit_5_nadine_circles', (
    'NB00 Nadine reference manifest required; rerun NB00 Block 10')
TREE_FILES={site:Path(p) for site,p in A['shared_annotations'].items()}
ROI_FILE=Path(A['roi'])
CIRCLE_FILE=Path(A['exhaustive_annotations']['complete_roi_boundaries'])
EXH_FILE=Path(A['exhaustive_annotations']['positive_polygons'])
SITES=['Baego','MOS','Yangambi','Yoko']
EXPECTED_CIRCLES=int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])
EXPECTED_ROWS=set(map(int,A['exhaustive_annotations']['verified_exhaustive_circle_rows']))
assert EXPECTED_CIRCLES==34 and len(EXPECTED_ROWS)==34, (
    'This revision is pinned to the newly audited 34 Nadine circles')
assert CIRCLE_FILE.name.casefold()=='circles_roi_nadine.shp'
assert EXH_FILE.name.casefold()=='circles_annotations_nadine.shp'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert len(MOSAICS)==5 and len(TREE_FILES)==4
assert set(TREE_FILES)==set(SITES)
print('NEW REFERENCE CONTRACT: 30 retained circles; 5 mosaics / 4 forest groups. MOS 3-4 excluded.')
FOLDS=3;PATCH=512;BUFFER_M=15.;SEED=20261015
for p in [ROI_FILE,CIRCLE_FILE,EXH_FILE,*TREE_FILES.values(),*[Path(r['path']) for r in MOSAICS.values()]]:
    assert p.exists(),p
print('Found',len(MOSAICS),'RGB mosaics and',len(TREE_FILES),'shared tree annotation files')
print('Read-only input:',DATA,'\nWrites:',OUT)


## 02 — Repair geometry individually, with traceable quarantines

Invalid geometries repaired in memory if relative area change <= 5% for partial tree polygons and <= 1% for ROI/exhaustive reference polygons; otherwise excluded individually, never all polygons from the same file. Review quarantines before relying on the final outputs.


In [ ]:
REPAIRS=[]
def parts(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in parts(q)]
    return []
def load_polygons(path):
    max_change=.05 if Path(path) in TREE_FILES.values() else .01
    raw=gpd.read_file(path)
    assert raw.crs is not None,path
    rows=[];geoms=[]
    for idx,g in enumerate(raw.geometry):
        status='OK';delta=0.
        if g is None or g.is_empty:status='EMPTY'
        elif not g.is_valid or g.geom_type not in ('Polygon','MultiPolygon'):
            try:
                fixed=unary_union(parts(make_valid(g)))
                delta=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if not fixed.is_valid or fixed.is_empty or delta>max_change:status='QUARANTINE'
                else:g=fixed;status='REPAIRED'
            except Exception:status='QUARANTINE'
        REPAIRS.append(dict(file=path.name,row=idx,status=status,area_change=delta))
        if status in ('OK','REPAIRED'):rows.append(idx);geoms.append(g)
    output=raw.iloc[rows].copy().reset_index(drop=True)
    output.geometry=geoms
    output['_row_original']=rows
    print(path.name,':',len(output),'kept,',len(raw)-len(output),'quarantined')
    return output
ROI=load_polygons(ROI_FILE);CIRCLES=load_polygons(CIRCLE_FILE)
EXH=load_polygons(EXH_FILE)
TREES={site:load_polygons(path) for site,path in TREE_FILES.items()}
pd.DataFrame(REPAIRS).to_csv(OUT/'01_geometry_repairs.csv',index=False)
assert ROI.geometry.is_valid.all()
for site,g in TREES.items():
    assert 'Class_ID' in g.columns,site
    cls=pd.to_numeric(g.Class_ID,errors='coerce')
    g['_label']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1)).astype('int8')
    print(site,':',int((g._label==1).sum()),'Musanga /',int((g._label==0).sum()),'known negatives')
# As in Notebook 00: candidate circles with no Musanga are not assumed exhaustive.
exh_union=unary_union(EXH.to_crs(CIRCLES.crs).geometry.to_list())
verified=[i for i,g in enumerate(CIRCLES.geometry) if g.intersection(exh_union).area>1]
assert set(CIRCLES['_row_original'].iloc[verified].astype(int))==EXPECTED_ROWS, (
    'NB00 and NB01 disagree on verified circle rows; do not mix references')
CIRCLES=CIRCLES.iloc[verified].copy().reset_index(drop=True)
CIRCLES['circle_id']=CIRCLES['_row_original'].astype(int)
assert len(CIRCLES)==EXPECTED_CIRCLES,(
    f'Expected {EXPECTED_CIRCLES} verified Nadine circles; got {len(CIRCLES)}')
assert len(EXH)==int(A['exhaustive_annotations']['musanga_polygons']), (
    'Polygon count changed or repairs were quarantined; inspect before proceeding')
assert set(CIRCLES.circle_id)==EXPECTED_ROWS
print('All',len(EXH),'exhaustive polygons retained, regardless of Class_ID')
print('Verified raw exhaustive circles:',list(CIRCLES.circle_id))
assert EXCLUDED_IDS.issubset(set(CIRCLES.circle_id))
CIRCLES=CIRCLES[~CIRCLES.circle_id.isin(EXCLUDED_IDS)].copy()
assert len(CIRCLES)==30
print('30 retained circles; excluded:',sorted(EXCLUDED_IDS))


## 03 — Six mosaics nested within four forest groups; hard general ROI


In [ ]:
REG=[];GEO={}
for name,info in MOSAICS.items():
    with rasterio.open(info['path']) as ds:
        assert ds.crs and ds.crs.is_projected and ds.count>=3,name
        footprint=box(*ds.bounds)
        roi=ROI.to_crs(ds.crs)
        eligible=unary_union(roi.geometry.to_list()).intersection(footprint)
        REG.append(dict(mosaic=name,site=info['site'],roi_overlap_m2=float(eligible.area),
                        width=ds.width,height=ds.height,pixel_m=abs(ds.res[0]),crs=str(ds.crs)))
        GEO[name]=dict(transform=ds.transform,crs=ds.crs,width=ds.width,height=ds.height,
                       footprint=footprint,roi=roi,site=info['site'])
REG=pd.DataFrame(REG);display(REG)
REG.to_csv(OUT/'01_mosaic_registry.csv',index=False)
assert (REG.roi_overlap_m2>0).all(),'At least one mosaic lacks ROI coverage: resolve ROI.shp before continuing'


## 04 — Circle-to-mosaic ownership and spatial CV

First audit all 34 Nadine circles against **both** each orthomosaic footprint and general ROI.shp. NB00 tested image bounding boxes; a circle can be fully inside imagery but outside the modelling ROI. Save a full diagnostic and stop if any circle is under 95% covered. Only after resolving discrepancies, build three spatial folds per forest group.


In [ ]:
# Audit footprint coverage separately from general ROI coverage, for EVERY circle.
# NB00 checked orthomosaic bounding-box overlap; that does NOT prove ROI coverage.
roi_unions={name:unary_union(d['roi'].geometry.to_list()) for name,d in GEO.items()}
LINK_ROWS=[]
for _,row in CIRCLES.iterrows():
    candidates=[]
    for name,d in GEO.items():
        g=gpd.GeoSeries([row.geometry],crs=CIRCLES.crs).to_crs(d['crs']).iloc[0]
        bbox_area=float(g.intersection(d['footprint']).area)
        in_roi_area=float(g.intersection(d['footprint']).intersection(roi_unions[name]).area)
        candidates.append((bbox_area,in_roi_area,name))
    # Assign the mosaic by image-footprint overlap, NOT ROI overlap.
    # This identifies circles that have imagery but lie outside ROI.shp.
    bbox_area,roi_area,name=max(candidates,key=lambda v:(v[0],v[1]))
    circle_area=float(row.geometry.area)
    LINK_ROWS.append(dict(circle_id=int(row.circle_id),site=GEO[name]['site'],mosaic=name,
        circle_area_m2=circle_area,imagery_bbox_area_m2=bbox_area,
        imagery_bbox_coverage_fraction=bbox_area/circle_area,
        roi_intersection_area_m2=roi_area,roi_coverage_fraction=roi_area/circle_area,
        cover_pct=100*row.geometry.intersection(exh_union).area/circle_area))
COVERAGE=pd.DataFrame(LINK_ROWS)
COVERAGE.to_csv(OUT/'01_all_circle_roi_coverage_diagnostic.csv',index=False)
print('=== RETAINED 30-CIRCLE ROI AND MOSAIC COVERAGE DIAGNOSTIC ===')
display(COVERAGE.round(4))
problem=COVERAGE[(COVERAGE.imagery_bbox_coverage_fraction<.95) |
                 (COVERAGE.roi_coverage_fraction<.95)].copy()
print('=== CIRCLES REQUIRING ATTENTION (<95% coverage) ===')
display(problem.round(4))
print('Diagnostic saved:',OUT/'01_all_circle_roi_coverage_diagnostic.csv')
if len(problem):
    print('These circles must not be silently dropped or assigned negative outside ROI.shp.')
    raise AssertionError(
        'Reference circles have insufficient image and/or general ROI coverage: '+
        str(problem.circle_id.tolist())+
        '. Inspect ROI.shp against Nadine circle boundaries. '
        'NB00 bounding-box overlap alone was insufficient.')
LINK=COVERAGE[['circle_id','site','mosaic','roi_coverage_fraction','cover_pct']].copy()
expected_site_counts={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
assert LINK.groupby('site').size().to_dict()==expected_site_counts,(
    'New circle ownership unexpectedly differs from audited 34-circle dataset')
assert LINK.circle_id.is_unique and len(LINK)==30
fold_rows=[]
for site,rows in LINK.groupby('site'):
    ids=rows.circle_id.to_list()
    centers=np.array([[CIRCLES.loc[CIRCLES.circle_id==j].geometry.iloc[0].centroid.x,
                       CIRCLES.loc[CIRCLES.circle_id==j].geometry.iloc[0].centroid.y] for j in ids])
    assert CIRCLES.crs.is_projected
    seeds=[int(np.argmax(np.linalg.norm(centers-centers.mean(axis=0),axis=1)))]
    while len(seeds)<FOLDS:
        dist=np.linalg.norm(centers[:,None,:]-centers[seeds][None,:,:],axis=2).min(axis=1)
        dist[seeds]=-1
        seeds.append(int(np.argmax(dist)))
    for ix,j in enumerate(ids):
        fold=int(np.argmin(np.linalg.norm(centers[ix]-centers[seeds],axis=1)))
        fold_rows.append(dict(circle_id=j,fold=fold))
LINK=LINK.merge(pd.DataFrame(fold_rows),on='circle_id',validate='one_to_one')
assert LINK.groupby('site').fold.nunique().eq(FOLDS).all()
assert LINK.groupby(['site','fold']).circle_id.nunique().ge(1).all()
assert LINK.circle_id.nunique()==30
print('Per-site / fold circle counts:')
display(LINK.groupby(['site','fold']).circle_id.nunique().unstack(fill_value=0))
display(LINK.sort_values(['site','fold','circle_id']).round(3))
LINK.to_csv(OUT/'01_circle_spatial_folds.csv',index=False)


## 05 — Single-source raster label generator (native resolution)

`label_window()` returns target, loss weight, annotation source, uncertainty and general ROI masks for a **small native raster window**. `Class_ID=1` means Musanga; other numeric classes mean known non-Musanga. Unknown background outside circles has zero weight. Explicit contradictions have zero training weight. For evaluation, only exhaustive held-out circle pixels count, and contradictions are retained in the full-reference analysis.


In [ ]:
CACHE={}
for name,d in GEO.items():
    circles=CIRCLES.to_crs(d['crs']).copy()
    circles['assigned_mosaic']=circles.circle_id.map(LINK.set_index('circle_id').mosaic)
    circles['fold']=circles.circle_id.map(LINK.set_index('circle_id').fold)
    d['circles']=circles[circles.assigned_mosaic==name].copy()
    # Spatial protection covers all circles of this forest group,
    # even where MOS orthomosaics overlap.
    d['site_circles']=circles[circles.circle_id.isin(
        LINK.loc[LINK.site==d['site'],'circle_id'])].copy()
    d['site_circles']['fold']=d['site_circles'].circle_id.map(
        LINK.set_index('circle_id').fold)
    d['positive_exhaustive']=EXH.to_crs(d['crs'])
    d['trees']=TREES[d['site']].to_crs(d['crs'])
    CACHE[name]=d
def label_window(name,win,fold=None,role='production'):
    """role='train', 'val', 'production'. Valid pixels only inside ROI.shp."""
    d=CACHE[name]
    shape=(int(win.height),int(win.width))
    transform=rasterio.windows.transform(win,d['transform'])
    bounds=box(*rasterio.windows.bounds(win,d['transform']))
    def burn(geoms):
        intersecting=[g.intersection(bounds) for g in geoms if g is not None and g.intersects(bounds)]
        if not intersecting:return np.zeros(shape,dtype=bool)
        return rasterize([(g,1) for g in intersecting if not g.is_empty],
            out_shape=shape,transform=transform,fill=0,dtype='uint8').astype(bool)
    roi=burn(d['roi'].geometry)
    circles=d['circles']
    if role in ('train','val'):
        assert fold in range(FOLDS)
        circles=circles[circles.fold.ne(fold) if role=='train' else circles.fold.eq(fold)]
    elif role!='production':raise ValueError(role)
    exhaustive=burn(circles.geometry)&roi
    pos_ex=burn(d['positive_exhaustive'].geometry)&exhaustive
    trees=d['trees'];pos_tree=burn(trees.loc[trees._label==1].geometry)
    neg_tree=burn(trees.loc[trees._label==0].geometry)
    uncertain=(pos_ex&neg_tree)|(exhaustive&~pos_ex&pos_tree)|(~exhaustive&pos_tree&neg_tree)
    source=np.zeros(shape,dtype='uint8')
    source[(pos_tree|neg_tree)&~exhaustive]=1
    source[exhaustive]=2
    target=(pos_ex | (~exhaustive&pos_tree)).astype('uint8')
    if role=='val':
        weight=exhaustive.astype('uint8') # full reference includes disagreement for independent QC
    else:
        weight=(roi&(source>0)&~uncertain).astype('uint8')
        if role=='train':
            # also suppress labels within buffer of held-out circles
            holdout=d['site_circles'][d['site_circles'].fold==fold]
            forbidden=burn([g.buffer(BUFFER_M) for g in holdout.geometry])
            weight[forbidden]=0
    weight[~roi]=0;source[~roi]=0;target[~roi]=0
    return dict(target=target,weight=weight,source=source,
                uncertain=(uncertain&roi).astype('uint8'),roi=roi.astype('uint8'))
print('Prepared reference rasterization function for',len(CACHE),'mosaics')


## 06 — Compact patch-window candidates

Systematic windows cover exhaustive circles; partial crown windows provide supplementary training examples away from exhaustive circles. **No full-resolution RGB mosaics are copied or rasterized.** Each window points to the original TIFF. MOS crowns are assigned to one mosaic when image footprints overlap.


In [ ]:
PATCH_ROWS=[]
MAX_PARTIAL_PER_MOSAIC=300
for name,d in CACHE.items():
    W,H=d['width'],d['height']
    assert W>=PATCH and H>=PATCH
    for _,circle in d['circles'].iterrows():
        w=from_bounds(*circle.geometry.bounds,transform=d['transform'])
        r0=max(0,int(w.row_off));c0=max(0,int(w.col_off))
        r1=min(H,int(math.ceil(w.row_off+w.height)));c1=min(W,int(math.ceil(w.col_off+w.width)))
        for rr in range((r0//PATCH)*PATCH,r1,PATCH):
            for cc in range((c0//PATCH)*PATCH,c1,PATCH):
                win=Window(cc,rr,min(PATCH,W-cc),min(PATCH,H-rr))
                if not circle.geometry.intersects(box(*rasterio.windows.bounds(win,d['transform']))):continue
                PATCH_ROWS.append(dict(site=d['site'],mosaic=name,kind='exhaustive',
                    circle_id=int(circle.circle_id),fold=int(circle.fold),
                    row=int(rr),col=int(cc),height=int(win.height),width=int(win.width)))
    # Partial-label patches cannot overlap circles or their exclusion buffers.
    exclusion=unary_union([g.buffer(BUFFER_M) for g in d['site_circles'].geometry]) if len(d['site_circles']) else None
    trees=d['trees'][d['trees']._label.isin([0,1])]
    rng=np.random.default_rng(SEED+sum(map(ord,name)))
    seen=set()
    for i in rng.permutation(len(trees)):
        p=trees.geometry.iloc[i].representative_point()
        if d['site']=='MOS':
            owners=sorted(k for k,v in CACHE.items() if v['site']=='MOS' and v['footprint'].covers(p))
            if owners and name!=owners[0]:continue
        rr,cc=rasterio.transform.rowcol(d['transform'],p.x,p.y)
        rr=int(max(0,min(H-PATCH,int(rr)-PATCH//2)))
        cc=int(max(0,min(W-PATCH,int(cc)-PATCH//2)))
        key=(rr//(PATCH//2),cc//(PATCH//2))
        if key in seen:continue
        win=Window(cc,rr,PATCH,PATCH)
        shape=box(*rasterio.windows.bounds(win,d['transform']))
        if exclusion is not None and shape.intersects(exclusion):continue
        if not any(g.intersects(shape) for g in d['roi'].geometry):continue
        seen.add(key)
        PATCH_ROWS.append(dict(site=d['site'],mosaic=name,kind='partial',circle_id=-1,fold=-1,
                               row=rr,col=cc,height=PATCH,width=PATCH))
        if len(seen)>=MAX_PARTIAL_PER_MOSAIC:break
PATCHES=pd.DataFrame(PATCH_ROWS).drop_duplicates()
assert len(PATCHES)>0
display(PATCHES.groupby(['site','kind'],as_index=False).size())
PATCHES.to_csv(OUT/'01_candidate_windows.csv',index=False)
print('Reference windows:',len(PATCHES),'(RGB remains in its original TIFFs)')


## 07 — Fold-specific leak-proof train/validation manifests

Entire RGB training windows crossing a held-out circle plus 15 m buffer are excluded, even if the labels outside the circle were usable. The test region stays independent of partial annotation examples. Held-out circle windows are evaluation windows only; their reference pixels are confined to the specific held-out circle.


In [ ]:
ROWS=[]
for fold in range(FOLDS):
    for _,p in PATCHES.iterrows():
        d=CACHE[p.mosaic]
        win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
        footprint=box(*rasterio.windows.bounds(win,d['transform']))
        holdout=d['site_circles'][d['site_circles'].fold==fold]
        near_holdout=any(footprint.intersects(g.buffer(BUFFER_M)) for g in holdout.geometry)
        if p.kind=='exhaustive' and int(p.fold)==fold:split='val'
        elif near_holdout:split='excluded_spatial_guard'
        else:split='train'
        ROWS.append(dict(cv_fold=fold,site=p.site,mosaic=p.mosaic,kind=p.kind,
                         circle_id=int(p.circle_id),row=int(p.row),col=int(p.col),
                         width=int(p.width),height=int(p.height),split=split))
SPLITS=pd.DataFrame(ROWS)
# Every site has held-out exhaustive reference in each fold.
counts=SPLITS[SPLITS.split=='val'].groupby(['cv_fold','site']).circle_id.nunique()
assert len(counts)==FOLDS*len(SITES) and counts.min()>=1,counts
for _,row in SPLITS[SPLITS.split=='train'].iterrows():
    d=CACHE[row.mosaic]
    geom=box(*rasterio.windows.bounds(Window(row.col,row.row,row.width,row.height),d['transform']))
    assert not any(geom.intersects(g.buffer(BUFFER_M)) for g in
                   d['site_circles'].loc[d['site_circles'].fold==row.cv_fold].geometry),'Training RGB spatial leakage'
display(SPLITS.groupby(['cv_fold','site','split'],as_index=False).size())
SPLITS.to_csv(OUT/'01_cv_window_manifest.csv',index=False)


## 08 — Exhaustive annotation QA and central-crop coverage

Select high-coverage tiles (not arbitrary row order) for visualization. Audit whether the central 256 px crop used by NB02D actually intersects each reference circle; report circles not represented in any central crop.


In [ ]:
# Block 08 — Diagnostic QC: select a genuinely representative 512 px circle window.
# NB02D trains on the CENTRAL 256 px crop of each 512 px candidate.
# Hence measure both full-window and central-crop annotation support.
from matplotlib.colors import ListedColormap
QC_ROWS=[]
circle_lookup=CIRCLES.set_index('circle_id')
for p in PATCHES.loc[PATCHES.kind=='exhaustive'].itertuples(index=False):
    geom=circle_lookup.loc[int(p.circle_id)].geometry
    d=CACHE[p.mosaic]
    # Circle and raster share CRS only after transforming to this mosaic.
    circle_local=gpd.GeoSeries([geom],crs=CIRCLES.crs).to_crs(d['crs']).iloc[0]
    win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
    full_box=box(*rasterio.windows.bounds(win,d['transform']))
    crop_size=min(256,int(win.height),int(win.width))
    center_win=Window(int(win.col_off)+(int(win.width)-crop_size)//2,
                      int(win.row_off)+(int(win.height)-crop_size)//2,
                      crop_size,crop_size)
    central_box=box(*rasterio.windows.bounds(center_win,d['transform']))
    full_area=float(circle_local.intersection(full_box).area)
    central_area=float(circle_local.intersection(central_box).area)
    QC_ROWS.append(dict(site=p.site,mosaic=p.mosaic,circle_id=int(p.circle_id),
        row=int(p.row),col=int(p.col),height=int(p.height),width=int(p.width),
        circle_area_full_m2=full_area,circle_area_center256_m2=central_area))
QC_WINDOWS=pd.DataFrame(QC_ROWS)
QC_WINDOWS['usable_center256']=QC_WINDOWS.circle_area_center256_m2.gt(0)
coverage_summary=QC_WINDOWS.groupby('site').agg(
    candidate_windows=('circle_id','size'),
    windows_with_center256_circle=('usable_center256','sum'),
    distinct_circles=('circle_id','nunique'))
coverage_summary['circles_with_center256_circle']=(
    QC_WINDOWS[QC_WINDOWS.usable_center256].groupby('site').circle_id.nunique()
).reindex(coverage_summary.index,fill_value=0).astype(int)
display(coverage_summary)
missing_circles=(set(CIRCLES.circle_id.astype(int))-
                 set(QC_WINDOWS.loc[QC_WINDOWS.usable_center256,'circle_id'].astype(int)))
print('Circles without ANY candidate window containing reference in central 256px:',sorted(missing_circles))
QC_WINDOWS.to_csv(OUT/'01_exhaustive_window_center256_QC.csv',index=False)
# Choose high-coverage tiles for the actual visual QC, rather than arbitrary median rows.
examples=[]
for site in SITES:
    group=QC_WINDOWS[(QC_WINDOWS.site==site)&(QC_WINDOWS.usable_center256)]
    assert len(group)>0,f'No representative exhaustive window for site {site}'
    examples.append(group.sort_values(
        ['circle_area_center256_m2','circle_area_full_m2'],ascending=False).iloc[0])
fig,axs=plt.subplots(len(examples),3,figsize=(12,3.3*len(examples)),constrained_layout=True)
if len(examples)==1:axs=axs[None,:]
for i,p in enumerate(examples):
    win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
    lab=label_window(p.mosaic,win,role='production')
    with rasterio.open(MOSAICS[p.mosaic]['path']) as ds:
        rgb=np.moveaxis(ds.read([1,2,3],window=win,masked=True).filled(0),0,-1)
    if rgb.dtype!=np.uint8:
        scale=max(float(np.nanpercentile(rgb,99)),1.)
        rgb=(255*np.clip(rgb.astype('float32')/scale,0,1)).astype('uint8')
    mask=lab['weight']>0
    n_trusted=int(mask.sum())
    n_positive=int(((lab['target']>0)&mask).sum())
    assert n_trusted>250,f'No labelled pixels in selected QC tile: {p.site}, circle {p.circle_id}'
    print(f'{p.site}: circle {int(p.circle_id)}; trusted {n_trusted:,} pixels; '
          f'Musanga {n_positive:,} ({100*n_positive/n_trusted:.1f}%)')
    category=np.zeros_like(lab['target'])
    category[mask]=1
    category[mask&(lab['target']==1)]=2
    category[lab['uncertain']>0]=3
    category[lab['roi']==0]=4
    axs[i,0].imshow(rgb);axs[i,0].set_title(f'{p.site}: RGB (circle {int(p.circle_id)})')
    axs[i,1].imshow(category,vmin=0,vmax=4,cmap=ListedColormap(
        ['#333333','#cccccc','#2b8c4b','#efb63e','#000000']))
    axs[i,1].set_title('Unknown / non-Musanga / Musanga / conflict / outside ROI')
    axs[i,2].imshow(lab['weight'],cmap='gray',vmin=0,vmax=1)
    axs[i,2].set_title(f'Trusted: {100*mask.mean():.1f}% of tile')
    for ax in axs[i]:ax.axis('off')
fig.savefig(OUT/'01_label_QC.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)
if missing_circles:
    warnings.warn('Some circles have no exhaustive-window central-256 coverage. '
                  'Investigate candidate-window geometry before running NB02D.')


## 09 — Site and fold balance audit

The exhaustive circles are the reference for fraction accuracy. Their observed cover is reported separately by forest and CV fold. The 34 Nadine circles are not assumed to be a representative probability sample of each entire forest.


In [ ]:
STAT=LINK[['site','mosaic','circle_id','fold','roi_coverage_fraction','cover_pct']].copy()
display(STAT.groupby('site').agg(circles=('circle_id','size'),
    median_cover_pct=('cover_pct','median'),min_cover_pct=('cover_pct','min'),
    max_cover_pct=('cover_pct','max'),min_roi_coverage=('roi_coverage_fraction','min')).round(3))
display(STAT.groupby(['site','fold']).agg(circles=('circle_id','size'),
    mean_cover_pct=('cover_pct','mean')).round(2))
STAT.to_csv(OUT/'01_reference_balance.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4))
for ix,site in enumerate(SITES):
    vals=STAT.loc[STAT.site==site,'cover_pct'].to_numpy()
    ax.scatter(np.full(len(vals),ix)+np.linspace(-.12,.12,len(vals)),vals)
ax.set_xticks(range(len(SITES)),SITES);ax.set_ylabel('Exhaustive circle Musanga cover (%)')
ax.set_title('Within-site reference range (each point = one circle)')
ax.grid(axis='y',alpha=.2)
fig.savefig(OUT/'01_cover_by_site.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)


## 10 — Data contract and model-development hand-off

Notebook 02 should compare exhaustive-only and exhaustive-plus-partial supervision, using **the same folds**. Select pooled versus site-adapted U-Nets based on out-of-fold predictions, fraction bias and calibration, not only segmentation F1. Then refit on all labels for production probability maps **within ROI.shp only**. Include an independent valid RGB/nodata mask while reading TIFF windows.

The target rasterization function is part of this version-pinned notebook; do not redefine label precedence silently.


In [ ]:
contract=dict(schema='musanga_v3_reference_01_no_MOS34',input_manifest=str(manifest_path),
  site_groups=SITES,mosaics=list(MOSAICS),circle_count=len(CIRCLES),
  folds=FOLDS,seed=SEED,patch_px=PATCH,buffer_m=BUFFER_M,
  circle_reference_generation='Nadine_30_no_MOS34',excluded_mosaic=EXCLUDED_MOSAIC,excluded_circle_ids=sorted(EXCLUDED_IDS),
  circle_ids=list(map(int,sorted(CIRCLES.circle_id))),
  exhaustive_all_polygons_musanga=True,
  output_directory=str(OUT),
  general_roi=str(ROI_FILE),exhaustive_roi=str(CIRCLE_FILE),
  exhaustive_positive=str(EXH_FILE),partial_crowns={s:str(p) for s,p in TREE_FILES.items()},
  label_policy=dict(
    exhaustive='positive polygons Musanga; remaining confirmed background non-Musanga',
    partial='labelled crowns only; blank unknown',unknown='weight zero',
    disagreement='explicit contradictions ignored during training, retained for full-reference validation',
    production='train with all usable annotations after spatial CV selection; never predict outside general ROI'),
  cv_policy='three within-forest spatial folds, MOS treated as one forest group; 15m RGB window exclusion',
  evaluation=['pixel AUC/AP/Brier','circle mean-probability fraction bias/MAE',
              'thresholded cover bias','spatial F1/IoU','site-macro scores'],
  files=['01_geometry_repairs.csv','01_mosaic_registry.csv','01_circle_spatial_folds.csv',
         '01_candidate_windows.csv','01_cv_window_manifest.csv',
         '01_reference_balance.csv','01_label_QC.png'])
(OUT/'01_data_contract.json').write_text(json.dumps(contract,indent=2))
print('Notebook 01 complete. Outputs:')
for f in sorted(OUT.glob('01_*')):print(' ',f.name)
print('IMPORTANT: new reference outputs are isolated; adapt NB02D REF before fitting.')
print('No training, full-mosaic raster copying or inference performed.')


## 11 — Ground-measured Musanga plots (external validation registry)

Read `Annotations/Plot_Data.shp` and `pct_musang`. Preserve **all** observations, including those outside the new general ROI. Report per-plot geometry validity, per-mosaic footprint overlaps, fractional ROI overlap, and whether a candidate training window intersects its geometry. No field observation is used as a supervised target, model-selection input, or calibration observation. **Important:** for point plots, a point is not the actual field-plot footprint. Until the field plot's size and meaning of `pct_musang` are documented, we only flag training-window proximity; we do not impose an arbitrary exclusion buffer.

This addition does **not** modify previous folds, windows, or M1/M2 checkpoints. It is a standalone audit block and can be rerun after loading previous NB01 state.

In [ ]:
# Block 11 — Register field measurements as RESERVED EXTERNAL VALIDATION.
PLOT_FILE=DATA/'Annotations'/'Plot_Data.shp'
assert PLOT_FILE.is_file(),f"New plot file missing: {PLOT_FILE}"
PLOTS=gpd.read_file(PLOT_FILE)
assert len(PLOTS)>0 and PLOTS.crs is not None,"Field plots need rows and a defined CRS"
assert 'pct_musang' in PLOTS.columns,("Missing pct_musang",list(PLOTS.columns))
pct=pd.to_numeric(PLOTS.pct_musang,errors='coerce')
plot_invalid=PLOTS.geometry.isna()|PLOTS.geometry.is_empty|~PLOTS.geometry.is_valid
plot_table=[]
for i,g in enumerate(PLOTS.geometry):
    row=dict(plot_row=int(i),pct_musang=float(pct.iloc[i]) if pd.notna(pct.iloc[i]) else np.nan,
        valid_pct=bool(pd.notna(pct.iloc[i]) and 0<=pct.iloc[i]<=100),
        valid_geometry=bool(not plot_invalid.iloc[i]),
        geometry_type=str(PLOTS.geom_type.iloc[i]),in_roi=False,roi_fraction=np.nan,
        mosaics="",sites="",touching_candidate_windows=0)
    if plot_invalid.iloc[i]:
        plot_table.append(row);continue
    roi_g=ROI.to_crs(PLOTS.crs)
    roi_union=unary_union(roi_g.geometry.to_list())
    row['in_roi']=bool(g.intersects(roi_union))
    row['roi_fraction']=float(g.intersection(roi_union).area/g.area) if g.area>0 else float(roi_union.covers(g))
    present=[];sites=set();touch=0
    for name,d in GEO.items():
        local=gpd.GeoSeries([g],crs=PLOTS.crs).to_crs(d['crs']).iloc[0]
        if not local.intersects(d['footprint']):continue
        present.append(name);sites.add(d['site'])
        windows=PATCHES[PATCHES.mosaic==name]
        for w in windows.itertuples(index=False):
            window=Window(int(w.col),int(w.row),int(w.width),int(w.height))
            if local.intersects(box(*rasterio.windows.bounds(window,d['transform']))):
                touch+=1
    row.update(mosaics=';'.join(present),sites=';'.join(sorted(sites)),
               touching_candidate_windows=touch)
    plot_table.append(row)
FIELD_PLOTS=pd.DataFrame(plot_table)
for idcol in ('plot_id','Plot_ID','id','ID','name','Name'):
    if idcol in PLOTS.columns and idcol not in FIELD_PLOTS:
        FIELD_PLOTS.insert(1,idcol,PLOTS[idcol].to_numpy())
FIELD_PLOTS['potential_uav_validation']=(
    FIELD_PLOTS.valid_pct&FIELD_PLOTS.valid_geometry&FIELD_PLOTS.in_roi&
    FIELD_PLOTS.mosaics.astype(str).ne(''))
FIELD_PLOTS.to_csv(OUT/'01_external_field_plot_registry.csv',index=False)
display(FIELD_PLOTS)
print('Field plots:',len(FIELD_PLOTS),'| inside/intersect ROI:',FIELD_PLOTS.in_roi.sum(),
      '| outside ROI retained:',(~FIELD_PLOTS.in_roi).sum(),
      '| candidate training-window intersections:',FIELD_PLOTS.touching_candidate_windows.sum())
if FIELD_PLOTS.touching_candidate_windows.sum()>0:
    warnings.warn('Some candidate training windows intersect field-plot geometry. '
                  'Before claiming independent spatial validation, confirm plot dimensions '
                  'and impose an exclusion zone before training future models.')
if PLOTS.geom_type.isin(['Point','MultiPoint']).any():
    warnings.warn('Point geometries cannot define plot-scale canopy-fraction support. '
                  'Supply plot dimensions before plot-versus-UAV aggregation.')
FIELD_CONTRACT=dict(file=str(PLOT_FILE),fraction_column='pct_musang',
    plot_count=int(len(FIELD_PLOTS)),outside_roi_count=int((~FIELD_PLOTS.in_roi).sum()),
    potential_uav_validation_count=int(FIELD_PLOTS.potential_uav_validation.sum()),
    fields=list(PLOTS.columns),crs=str(PLOTS.crs),
    registry_csv=str(OUT/'01_external_field_plot_registry.csv'),
    reserved_for_external_validation=True,
    training_protection='Not yet enforced: plot support dimensions required',
    note='No changes to 01 spatial folds, labels, window manifests or saved neural networks')
(OUT/'01_external_field_plots_contract.json').write_text(json.dumps(FIELD_CONTRACT,indent=2))
print('Saved independent field-plot registry and supplementary data contract.')
